# Document RAG System.

In [7]:
# install needed libraries to use
!pip -q install langchain langchain-google-genai langchain-community google-genai faiss-cpu tiktoken python-dotenv pypdf langchain-huggingface sentence-transformers

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [18]:
# import libraries
import os
from google.colab import userdata

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# set environment variables for google and huggingface

os.environ['GOOGLE_API_KEY'] = userdata.get("GOOGLE_API_KEY")
os.environ['HUGGINGFACEHUB_ACCESS_TOKEN'] = userdata.get("HUGGINGFACEHUB_ACCESS_TOKEN")

### Load keys.

In [8]:
# import necessary libraries

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI,GoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

print("OpenAI key loaded:", bool(openai_key))
# print("OpenAI key:", openai_key)

print("\nGemini key loaded:", bool(gemini_key))
# print("Gemini key:", gemini_key)


OpenAI key loaded: True

Gemini key loaded: True


### Test key with prompt.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=gemini_key
)

response = llm.invoke("Hello")
print(response)


E0000 00:00:1759185374.177116  102820 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []} id='run--95cf3202-9c6a-49d9-a562-0aa5e92f20c8-0' usage_metadata={'input_tokens': 2, 'output_tokens': 32, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}}
Hello! How can I help you today?


In [5]:
print(response.content)

Hello! How can I help you today?


## Load document.

In [ ]:
# load and read PDF file

load_document = PyPDFLoader("ChenZhang_cropmapping_ReviewPaper.pdf")
document = load_document.load()

In [13]:
len(document)

29

In [ ]:
# First page of PDF
print(document[0].page_content) 

Review
Remote sensing for crop mapping: A perspective on current and future 
crop-specific land cover data products
Chen Zhang
a , *
, Hannah Kerner
b
, Sherrie Wang
c
, Pengyu Hao
d
, Zhe Li
e
, Kevin A. Hunt
e
,  
Jonathon Abernethy
e
, Haoteng Zhao
f
, Feng Gao
f
, Liping Di
a , *
, Claire Guo
a , g
, Ziao Liu
a
,  
Zhengwei Yang
e
, Rick Mueller
e
, Claire Boryan
e
, Qi Chen
h
, Peter C. Beeson
i
, Hankui K. Zhang
j
,  
Yu Shen
j , k
a
Center for Spatial Information Science and Systems, George Mason University, Fairfax, VA 22030, USA
b
School of Computing and Augmented Intelligence, Arizona State University, Tempe, AZ 85281, USA
c
Department of Mechanical Engineering, Massachusetts Institute of Technology, Cambridge, MA 02139, USA
d
Food and Agriculture Organization of the United Nations, Viale delle Terme di Caracalla, 00153 Rome, Italy
e
U.S. Department of Agriculture, National Agricultural Statistics Service, Washington, DC 20250, USA
f
U.S. Department of Agriculture, Agricultur

In [15]:
# 10th page of PDF
print(document[9].page_content)

the WoS database, including the publication title, abstract, or keywords. 
In our survey, we found that many papers introduced, discussed, or cited 
CDL, but did not directly use the data in their experiments. Therefore, 
IC1 could ensure that CDL has been applied in the selected publications, 
rather than simply mentioning it in passing.
To narrow down the publications to those specifically related to 
remote sensing, IC2 states that the publication ’ s “ Category ” field in the 
WoS database must be labeled as “ remote sensing ” . However, many 
publications related to remote sensing were published in computer sci -
ence, agricultural, or multidisciplinary journals, which were not cate -
gorized as “ remote sensing ” . To include these publications in this 
review, we added a rule that requires the presence of certain terms, such 
as “ Remote Sensing ” , “ Earth observation ” , “ Landsat ” , “ Sentinel ” , or 
“ MODIS ” in any of the title, keywords, or abstract of the publication.
T

In [17]:
# 14th page of PDF
print(document[13].page_content)

et al., 2013 ). CDL data also have been used to delineate and stratify 
regions, such as U.S. soybean growing areas ( Song et al., 2017 ), which 
helps in understanding field size patterns for more effective agricultural 
resource management.
Training samples: Beyond a crop type map, CDL is widely utilized 
as an authoritative geospatial benchmark to support field-level crop 
spectral signature training. The ML models trained with high-confidence 
pixels in CDL and associated products (e.g., CSB, Confidence Layer) can 
be applied to extend land cover classification while adjusting for factors 
such as hemisphere seasonality and evolving farming trends, which is 
invaluable for global crop monitoring. As discussed in RQ2, ML and DL 
are the main technologies in remote sensing studies, which rely on high- 
quality training data. Due to the extensive crop-specific land cover in -
formation, CDL has been extensively used to label training samples in EO 
data. This enables the further super

In [18]:
# 16th page of PDF
print(document[15].page_content)

and Kerner, 2023 ). Ground-truthing involves physically visiting agri -
cultural fields and recording the type of crop growing in the field. This 
process is prohibitively expensive and logistically challenging for many 
organizations and regions.
Currently available public reference samples are largely regional in 
scope ( Dufourg et al., 2023 ; Kondmann et al., 2021 ). Recent work has 
proposed novel methods of collecting ground-truth crop labels that 
reduce the cost of data collection. Paliyam et al. (2021) proposed a 
method called Street2Sat that uses computer vision (CV) techniques to 
transform roadside images of fields collected with car- and motorcycle 
helmet-mounted cameras into geo-referenced crop type labels of those 
fields. d’Andrimont et al. (2022) used CV techniques to extract crop type 
and phenology information from street-level images of fields taken with 
car-mounted cameras in the Netherlands. Yan and Ryu (2021) and Soler 
et al. (2024) used DL models to automati

## Split texts.

In [21]:
# split into chunks

doc_split= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = doc_split.split_documents(document)

In [22]:
len(chunks)

268

In [23]:
# display 4th chunk
print(chunks[3].page_content)

crop mapping from the perspective of crop-specific land cover data by evaluating over 60 open-access opera -
tional products, archival crop type map datasets, single-crop extent map datasets, cropping pattern datasets, and 
crop mapping platforms and systems. Using the Cropland Data Layer (CDL) – one of the most widely used 
products with over 25 years of continuous monitoring of U.S. croplands – as a case study, we also conduct a 
systematic literature review on the application of crop type maps in remote sensing science. Our analysis syn -
thesizes 129 research articles through three core research questions: (1) What EO data are used with CDL; (2) 
What scientific problems and technologies are explored using CDL; and (3) What role does CDL play in remote 
sensing applications. Furthermore, we delve into the implications of our vision for new data products and 
propose emerging research topics, ranging from extending the spatiotemporal coverage of current data products


In [ ]:
# display 5th chunk
print(chunks[4].page_content)

and reduce the impacts of extreme weather events.
6. Cultural and Recreational Value: Many landforms have cultural
significance and provide opportunities for recreation and tourism. Proper
management enhances their aesthetic and cultural value, attracting visitors
and supporting local economies.
7. Community Engagement and Resilience: Involving local
communities in landform management fosters a sense of ownership and
responsibility. This engagement can lead to more effective and culturally
relevant management strategies, enhancing community resilience.
8. Urban Planning and Development: Effective landform
management supports sustainable urban development by ensuring that


In [77]:
embeds = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeds)

## Retrieval.

In [81]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [82]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d4f78937860>, search_kwargs={'k': 5})

In [83]:
retriever.invoke("what is the main topic of the document?")

[Document(id='340fea89-7484-42db-9332-1a5384d91e15', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'URP TERM PAPER.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='THE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI'),
 Document(id='d3a78201-0d44-47e2-9db6-1eb4cb0daefa', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'sour

## Augmentation.

In [84]:
llm_gen = GoogleGenerativeAI(model="models/gemini-1.5-flash")

E0000 00:00:1759060425.855264   47311 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [86]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context IS INSUFFICIENT, just say you don't know and probably need more information.

    {context}

    Question: {question}
    """,
    input_variables=["context","question"]
)

In [89]:
question = "Is the context of stars is mentioned in this document? If yes, then what was discussed?"
retrieved_docs = retriever.invoke(question)

In [90]:
retrieved_docs

[Document(id='0da02404-b6cf-460b-b701-8d9563f2c104', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'URP TERM PAPER.pdf', 'total_pages': 8, 'page': 6, 'page_label': '7'}, page_content='and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that'),
 Document(id='340fea89-7484-42db-9332-1a5384d91e15', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': 

In [91]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [92]:
context_text

'and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\n\uf0b7 Limited vegetation adapted to dry conditions\n\uf0b7 Large temperature variations between day and night\n\uf0b7 Examples: Sahara Desert (Africa), Atacama Desert (Chile), Thar Desert\n(India)\n\uf0b7\nManagement Strategies: Management of deserts focuses on water\nconservation techniques, such as rainwater harvesting and sustainable\nland use planning to prevent habitat degradation and promote\nbiodiversity.\n\nand reduce the impacts of extreme weather e

In [93]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [94]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context IS INSUFFICIENT, just say you don't know and probably need more information.\n\n    and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\n\uf0b7 Limited vegetation adapted to dry conditions\n\uf0b7 Large temperature variations between day and night\n\uf0b7 Examples: Sahara Desert (Africa), Atacama Desert (Chile), Thar Desert\n(India)\n\uf0b7\nManagement Strategies: Management of de

## Answer Generation.

In [ ]:
response = llm.invoke(final_prompt)

In [ ]:
response.content

"I don't know and probably need more information. The context of stars is not mentioned in this document."

## Build chain.

In [102]:
# import libraries for chain building
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def reformat_doc(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [106]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(reformat_doc),
    'question': RunnablePassthrough()
}
)

In [ ]:
parallel_chain.invoke('what are the future directions in landform management')

{'context': "OUTLINE\nI. Introduction\nA. Definition of Landform\nB. Importance of Landform Management\nII. Types of Landform\nA. Mountains\n1. Characteristics\n2. Management Strategies\nB. Plateaus\n1. Characteristics\n2. Management Strategies\nC. Valleys\n1. Characteristics\n2. Management Strategies\nD. Plains\n1. Characteristics\n2. Management Strategies\nE. Deserts\n1. Characteristics\n2. Management Strategies\nIII. Landform Management\nA. Sustainable Land Use Practices\nB. Soil Conservation Techniques\nC. Water Resource Management\nD. Biodiversity Conservation\nIV. Future Directions in Landform Management\nA. Climate Change and Landform\nB. Technological Advancements in Management\nC. Community Involvement and Stakeholder Engagement\nV. Conclusion\nA. Recap of Key Points\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\nI. Int

In [ ]:
parse = StrOutputParser()

In [109]:
main_chain = parallel_chain | prompt | llm | parse

In [113]:
print(main_chain.invoke("what are the future directions in landform management"))

Based on the provided transcript context, the future directions in landform management include:

*   **Technological Advancements in Management:** This involves incorporating technology and remote sensing techniques to enhance the study and management of diverse landforms.
*   **Community Involvement and Stakeholder Engagement:** Collaborating with local communities and stakeholders is key to developing effective landform conservation strategies.
*   The text also mentions conducting regular assessments and monitoring of landforms to detect any changes or threats early on.

While "Climate Change and Landform" is listed as a future direction in the outline, the provided text does not offer specific details or strategies related to it. Therefore, the context is insufficient to elaborate on this point.


In [114]:
print(main_chain.invoke("what are the future directions in landform management. List and explain them in one sentence each."))

I don't know and probably need more information. The provided text lists the future directions but does not explain them.
